# Task 2 – Quantitative Analysis with TA-Lib & PyNance
## Nova Financial Solutions | Predicting Price Moves with News Sentiment

**Branch:** `task-2`  
**Objective:** Load historical stock price data, compute financial technical indicators (SMA, EMA, RSI, MACD, Bollinger Bands), and compute risk/return metrics equivalent to PyNance outputs.

---
### Notebook Structure
1. Data Loading & Preparation
2. Data Quality Assessment
3. Technical Indicators (TA-Lib)
4. PyNance-equivalent Financial Metrics
5. Multi-panel Indicator Visualization
6. Cumulative Return Analysis
7. Multi-ticker Comparison
8. Summary

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

from src.technical_indicators import TechnicalAnalyzer
from scripts.data_loader import download_stock_data, load_stock_data

TICKERS = ['AAPL', 'AMZN', 'GOOG', 'META', 'MSFT', 'NVDA', 'TSLA']
START   = '2019-01-01'
END     = '2024-12-31'
print('✅ Imports successful')

In [ ]:
# ── Cell 2: Download / Load Price Data ────────────────────────────────────────
price_data = {}
for ticker in TICKERS:
    csv_path = f'../data/raw/{ticker}_price_data.csv'
    if os.path.exists(csv_path):
        price_data[ticker] = pd.read_csv(csv_path, parse_dates=['Date'])
        print(f'Loaded {ticker} from cache ({len(price_data[ticker]):,} rows)')
    else:
        print(f'Downloading {ticker} from yfinance…')
        df = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=False)
        df.reset_index(inplace=True)
        os.makedirs('../data/raw', exist_ok=True)
        df.to_csv(csv_path, index=False)
        price_data[ticker] = df
        print(f'  → {len(df):,} rows saved.')

print(f'\n✅ {len(price_data)} tickers loaded.')

---
## Section 1: Data Quality Assessment

In [ ]:
# ── Cell 3: Data Quality Report ───────────────────────────────────────────────
print('='*65)
print(f'  {"Ticker":<8} {"Rows":>8} {"Start":>12} {"End":>12} {"Missing":>10}')
print('='*65)
for ticker, df in price_data.items():
    df['Date'] = pd.to_datetime(df['Date'])
    missing = df[['Open','High','Low','Close','Adj Close','Volume']].isnull().sum().sum()
    print(f'  {ticker:<8} {len(df):>8,} {str(df["Date"].min().date()):>12} '
          f'{str(df["Date"].max().date()):>12} {missing:>10}')
print('='*65)
print()
print('⚠️  Data Quality Notes:')
print('  - Non-trading days (weekends, holidays) are already excluded by yfinance.')
print('  - Corporate actions (splits, dividends) are reflected in Adj Close.')
print('  - Always use Adj Close for return calculations to avoid look-ahead bias.')

---
## Section 2: Technical Indicators — Primary Analysis (AAPL)

In [ ]:
# ── Cell 4: Compute Indicators for AAPL ──────────────────────────────────────
PRIMARY = 'AAPL'
ta = TechnicalAnalyzer(price_data[PRIMARY], ticker=PRIMARY)
df_indicators = ta.compute_all_indicators()

print(f'\nSample rows with indicators:')
display_cols = ['Close','SMA_20','SMA_50','EMA_50','RSI_14','MACD','MACD_Signal','Daily_Return']
print(df_indicators[display_cols].tail(10).to_string())

In [ ]:
# ── Cell 5: Indicator Statistics ──────────────────────────────────────────────
print(f'\n── {PRIMARY} Indicator Summary Stats ──')
summary_cols = ['Close','SMA_20','EMA_50','RSI_14','MACD','Daily_Return']
print(df_indicators[summary_cols].describe().round(4).to_string())

# RSI signal breakdown
rsi = df_indicators['RSI_14'].dropna()
overbought  = (rsi > 70).sum()
oversold    = (rsi < 30).sum()
neutral_rsi = ((rsi >= 30) & (rsi <= 70)).sum()
print(f'\nRSI signals: Overbought(>70)={overbought} | '
      f'Oversold(<30)={oversold} | Neutral={neutral_rsi}')

In [ ]:
# ── Cell 6: Figure 1 – Multi-Panel Technical Dashboard ───────────────────────
ta.plot_indicators(save_path='../data/raw/fig_task2_indicators_AAPL.png')

---
## Section 3: PyNance-equivalent Financial Metrics

In [ ]:
# ── Cell 7: Financial Metrics ─────────────────────────────────────────────────
metrics = ta.compute_pynance_metrics()

# Interpretation
print('\n── Metric Interpretations ──')
print('Sharpe Ratio > 1.0  → Risk-adjusted returns are acceptable')
print('Sharpe Ratio > 2.0  → Very strong risk-adjusted performance')
print('Max Drawdown        → Worst peak-to-trough decline in the period')
print('Sortino Ratio       → Like Sharpe but only penalises downside volatility')

In [ ]:
# ── Cell 8: Cumulative Return ─────────────────────────────────────────────────
ta.plot_cumulative_return(save_path='../data/raw/fig_task2_cumreturn_AAPL.png')

---
## Section 4: Multi-Ticker Comparison

In [ ]:
# ── Cell 9: All Tickers – Financial Metrics Table ─────────────────────────────
all_metrics = []
analyzers   = {}
for ticker in TICKERS:
    ta_obj = TechnicalAnalyzer(price_data[ticker], ticker=ticker)
    ta_obj.compute_all_indicators()
    m = ta_obj.compute_pynance_metrics()
    m['ticker'] = ticker
    all_metrics.append(m)
    analyzers[ticker] = ta_obj

metrics_df = pd.DataFrame(all_metrics).set_index('ticker')
print('\n── Financial Metrics Comparison Across Tickers ──')
print(metrics_df.to_string())

In [ ]:
# ── Cell 10: Figure 2 – Correlation of Daily Returns ─────────────────────────
returns_all = pd.DataFrame({
    t: analyzers[t].df['Daily_Return'] for t in TICKERS
})

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(returns_all.corr(), dtype=bool))
sns.heatmap(
    returns_all.corr(), annot=True, fmt='.3f', cmap='RdYlGn',
    mask=mask, ax=ax, vmin=-1, vmax=1,
    linewidths=0.5, cbar_kws={'label': 'Pearson r'}
)
ax.set_title('Daily Return Correlation Matrix Across Tickers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/raw/fig_task2_return_corr.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 11: Figure 3 – Normalised Price Performance ─────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))
colors = plt.cm.tab10.colors
for i, ticker in enumerate(TICKERS):
    cum_ret = analyzers[ticker].df['Cum_Return'].dropna() * 100
    ax.plot(cum_ret.index, cum_ret.values, label=ticker,
            linewidth=1.5, color=colors[i])

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Cumulative Return Comparison (2019–2024)', fontsize=13, fontweight='bold')
ax.set_ylabel('Cumulative Return (%)')
ax.set_xlabel('Date')
ax.legend(ncol=4, fontsize=9)
plt.tight_layout()
plt.savefig('../data/raw/fig_task2_cumreturn_all.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('✅ Task 2 Quantitative Analysis Complete!')
print('Commit message suggestion: feat(indicators): add TA-Lib indicators and PyNance metrics for all tickers')